In [1]:
import os
import copy
import torch
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score

from sklearn.model_selection import train_test_split
from torch_geometric.utils import to_networkx
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import add_self_loops
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from types import SimpleNamespace
from sklearn.ensemble import IsolationForest
import itertools

warnings.filterwarnings('ignore')

In [2]:
df_features = pd.read_csv('dataset/elliptic_txs_features.csv', header=None)
df_edges = pd.read_csv("dataset/elliptic_txs_edgelist.csv")
df_classes =  pd.read_csv("dataset/elliptic_txs_classes.csv")
df_classes['class'] = df_classes['class'].map({'unknown': 2, '1': 1, '2': 0})

In [3]:
df_classes['class'].value_counts()

class
2    157205
0     42019
1      4545
Name: count, dtype: int64

In [4]:
df_merge = df_features.merge(df_classes, how='left', right_on="txId", left_on=0)
df_merge = df_merge.sort_values(0).reset_index(drop=True)

# mapping nodes to indices
nodes = df_merge[0].values
map_id = {j:i for i,j in enumerate(nodes)}

# mapping edges to indices
edges = df_edges.copy()
edges.txId1 = edges.txId1.map(map_id)
edges.txId2 = edges.txId2.map(map_id)
edges = edges.astype(int)
edge_index = np.array(edges.values).T
edge_index = torch.tensor(edge_index, dtype=torch.long).contiguous()

# weights for the edges are equal in case of model without attention
weights = torch.tensor([1] * edge_index.shape[1] , dtype=torch.float32)

# maping node ids to corresponding indexes
node_features = df_merge.drop(['txId'], axis=1).copy()
node_features[0] = node_features[0].map(map_id)

# store known and unknown nodes
classified_idx = node_features['class'].loc[node_features['class'] != 2].index
unclassified_idx = node_features['class'].loc[node_features['class'] == 2].index

# replace unkown class with 0, to avoid having 3 classes, this data/labels never used in training
# node_features['class'] = node_features['class'].replace(2, 0)

labels = node_features['class'].values
y_train = labels[classified_idx]

# drop indeces, class and temporal axes
node_features = torch.tensor(np.array(node_features.drop([0, 'class', 1], axis=1).values, dtype=np.float32), dtype=torch.float32)

In [5]:
elliptic_dataset = Data(x = node_features,
                        edge_index = edge_index,
                        x0 = node_features,
                        y = torch.tensor(labels, dtype=torch.float32))

In [2]:
config = SimpleNamespace(seed = 0,
                         learning_rate = 0.001,
                         weight_decay = 1e-5,
                         input_dim = 165,
                         output_dim = 64,
                         hidden_size = 64,
                         # checkpoints_dir = './models/elliptic_gnn',
                         device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

In [8]:
_, _, _, _, train_idx, valid_idx = train_test_split(node_features[classified_idx],
                                                    y_train,
                                                    classified_idx,
                                                    test_size=0.15,
                                                    random_state=config.seed,
                                                    stratify=y_train)

elliptic_dataset.train_idx = torch.tensor(train_idx, dtype=torch.long)
elliptic_dataset.val_idx = torch.tensor(valid_idx, dtype=torch.long)

In [9]:
torch.save(elliptic_dataset, 'graph_data.pt')

In [3]:
elliptic_dataset = torch.load('graph_data.pt')

In [32]:
class SAGEConv(MessagePassing):
    def __init__(self, in_channels, out_channels, n_dim):
        super().__init__(aggr='mean')  # 使用 mean 聚合
        self.lin = nn.Linear(in_channels + n_dim, out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, x0, edge_index):
        # x 是节点自身的特征（用于更新），x0 是邻居节点的特征（用于聚合）
        # edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        return self.propagate(edge_index, x=x, x0=x0)

    def message(self, x_j, x0_j):
        # 注意：这里我们只使用 x0_j 来传播邻居信息
        return x0_j

    def update(self, aggr_out, x):
        # x 是节点本身的特征，拼接后线性映射
        out = torch.cat([x, aggr_out], dim=1)
        return self.relu(self.lin(out))

class SAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, n_dim):
        super().__init__()
        self.atten = nn.Parameter(torch.randn(1, in_channels))
        self.dropout = 0.1
        self.sage_layers = nn.ModuleList()
        self.sage_layers.append(SAGEConv(in_channels, hidden_channels, n_dim))
        self.sage_layers.append(SAGEConv(hidden_channels, hidden_channels, n_dim))
        self.out = nn.Linear(hidden_channels, out_channels)

    def forward(self, data, corrupt=False):
        if corrupt:
            num_nodes = data.num_nodes
            # 生成随机顺序
            perm = torch.randperm(num_nodes)
            # 打乱特征
            data.x = data.x[perm]
            data.x0 = data.x0[perm]

        x, x0, edge_index = data.x, data.x0, data.edge_index

        x0 = x0 * self.atten
        x = x * self.atten

        for i, sage in enumerate(self.sage_layers):
            # F.dropout(x, p=self.dropout, training=self.training)
            x = sage(x, x0, edge_index)
            # x = F.relu(x)

        # x = self.out(x)
        return x

class POLICE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, n_dim):
        super(POLICE, self).__init__()
        self.encoder = SAGE(in_channels, hidden_channels, out_channels, n_dim)
        # self.discriminator = Discriminator(ndim_out)
        self.loss = nn.BCEWithLogitsLoss()

    def forward(self, data):
        positive = self.encoder(data, corrupt=False)
        negative = self.encoder(data, corrupt=True)

        
        
        #       summary = torch.sigmoid(positive.mean(dim=0))

        # positive = self.discriminator(positive, summary)
        # negative = self.discriminator(negative, summary)

        l1 = self.loss(positive[data.train_idx], torch.full_like(positive[data.train_idx], 1))
        l2 = self.loss(negative[data.train_idx], torch.full_like(negative[data.train_idx], 0))

        return l1 + l2

def train_evaluate(model, data, optimizer, epochs):
    num_epochs = epochs

    model.train()
    for epoch in range(num_epochs+1):
        # Training
        optimizer.zero_grad()
        loss = model(data)
        loss.backward()
        optimizer.step()
       
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Train Loss: {loss:.3f}')


    return model


In [33]:
torch.manual_seed(config.seed)

police_model = POLICE(config.input_dim, config.hidden_size, config.output_dim, config.input_dim).to(config.device)
data_train = elliptic_dataset.to(config.device)

optimizer = torch.optim.Adam(police_model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
# criterion = torch.nn.BCEWithLogitsLoss()


In [ ]:
train_evaluate(police_model,
               data_train,
               optimizer,
               1000)

In [20]:
torch.save(police_model.state_dict(),'police-64-64-400.pth')

In [7]:
police_model.load_state_dict(torch.load('police-64-64-50.pth'))

<All keys matched successfully>

In [26]:

print(type(data_train.val_idx))       # 显示类型（如 numpy.ndarray）
print(data_train.val_idx.dtype)       # 显示数据类型（如 bool、int64 等）
print(data_train.val_idx.shape)       # 显示形状（行数）
print(data_train.val_idx[:10])

<class 'torch.Tensor'>
torch.int64
torch.Size([6985])
tensor([ 15658,   4831, 105639,    387, 153119, 110802, 160718, 146428,  75453,
        171188], device='cuda:0')


In [38]:
all_emb = police_model.encoder(data_train)
all_emb = all_emb.detach().cpu().numpy()
df_all_emb = pd.DataFrame(all_emb)

all_y = data_train.y.detach().cpu().numpy()
df_all_y = pd.DataFrame(all_y)

train_mask = data_train.train_idx.detach().cpu().numpy() 
train_data = df_all_emb.iloc[train_mask]

test_mask = data_train.val_idx.detach().cpu().numpy() 
test_data = df_all_emb.iloc[test_mask]
test_labels = df_all_y.iloc[test_mask]

In [17]:
contamination = [0.001, 0.01, 0.04, 0.05, 0.1, 0.2]
import tqdm
import gc

In [39]:
from pyod.models.cblof import CBLOF
n_est = [2,3,5,7,9,10]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(train_data)
    y_pred = clf_if.predict(val_data)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 36/36 [00:11<00:00,  3.19it/s]

{'n_estimators': 5, 'con': 0.01}
0.49090169437656495
              precision    recall  f1-score   support

         0.0     0.9053    0.7671    0.8305      6303
         1.0     0.1071    0.2581    0.1513       682

    accuracy                         0.7174      6985
   macro avg     0.5062    0.5126    0.4909      6985
weighted avg     0.8273    0.7174    0.7642      6985



In [40]:
from pyod.models.hbos import HBOS

n_est = [5,10,15,20,25,30]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(train_data)
    y_pred = clf_if.predict(val_data)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 36/36 [00:12<00:00,  2.96it/s]

{'n_estimators': 25, 'con': 0.01}
0.5034703308949193
              precision    recall  f1-score   support

         0.0     0.9040    0.8520    0.8772      6303
         1.0     0.1072    0.1642    0.1297       682

    accuracy                         0.7848      6985
   macro avg     0.5056    0.5081    0.5035      6985
weighted avg     0.8262    0.7848    0.8042      6985

